# 02 — Feature Engineering
**AI Investment Manager Portfolio Platform**

Transforms raw Parquet tables from `01_data_collection` into a single
ML-ready feature store saved as `data/processed/features.parquet`.

### Features built in this notebook
| Category | Features |
|---|---|
| Returns | daily, 5d, 21d, 63d, 126d rolling returns |
| Volatility | 21d, 63d rolling std of daily returns (annualised) |
| Momentum | RSI-14, MACD, MACD signal, MACD histogram |
| Price structure | Bollinger bands, 52-week position, price vs SMA50/200 |
| Volume | volume ratio vs 20d average |
| Fundamentals | revenue growth, margins (fiscal-year aligned via merge_asof) |
| Categorical | sector, industry |
| Target | forward 21d return bucketed into buy / hold / sell |

### Key principle — no data leakage
All rolling features use only past data (`rolling()` looks backward by default).
Financials are joined with `merge_asof(..., direction="backward")` so each
price row only sees reports already published on that date.

## 0. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import datetime

import sys
sys.path.append("..")  # make src/ importable from notebooks/
from src.data_collection import load_all, get_logger, PROCESSED_DIR

warnings.filterwarnings("ignore")
logger = get_logger("feature_engineering")

# ── Output path ────────────────────────────────────────────────────────────
FEATURES_FILE = PROCESSED_DIR / "features.parquet"

# ── Target definition ──────────────────────────────────────────────────────
FORWARD_DAYS = 21   # predict 1-month ahead return
N_QUANTILES  = 3    # labels: -1 (sell), 0 (hold), 1 (buy)

# ── Rolling window sizes ───────────────────────────────────────────────────
VOL_SHORT     = 21
VOL_LONG      = 63
RSI_PERIOD    = 14
MACD_FAST     = 12
MACD_SLOW     = 26
MACD_SIGNAL   = 9
BB_WINDOW     = 20
BB_STD        = 2
VOLUME_WINDOW = 20

logger.info("Feature engineering config loaded")
print("Config ready.")

## 1. Load Raw Tables

Load all three Parquet files produced by `01_data_collection.ipynb`.
All downstream work reads from the feature store — never re-fetches from yfinance.

In [ ]:
data       = load_all()
history    = data["history"].copy()
info       = data["info"].copy()
financials = data["financials"].copy()

# Ensure correct dtypes
history["Date"]               = pd.to_datetime(history["Date"])
financials["FiscalYearEnd"]   = pd.to_datetime(financials["FiscalYearEnd"])

print(f"History    : {history.shape}")
print(f"Info       : {info.shape}")
print(f"Financials : {financials.shape}")
print(f"\nDate range : {history['Date'].min().date()} to {history['Date'].max().date()}")
print(f"Tickers    : {history['Ticker'].nunique()}")

## 2. Prototype on AAPL First

Always build and validate logic on a single ticker before scaling to all 498.
Bugs surface in seconds, not minutes.

In [ ]:
aapl = (
    history[history["Ticker"] == "AAPL"]
    .copy()
    .sort_values("Date")
    .reset_index(drop=True)
)
print(f"AAPL rows: {len(aapl)}")
print(aapl[["Date", "Open", "High", "Low", "Close", "Volume"]].tail())

## 3. Return Features

Daily return is the foundation — every other feature builds on it.
Rolling returns capture momentum at different time horizons:
- 5d = weekly, 21d = monthly, 63d = quarterly, 126d = semi-annual

In [ ]:
def add_return_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add return features to a single-ticker DataFrame sorted by Date.
    pct_change() and rolling() look backwards only — no leakage.
    """
    df = df.copy()
    df["daily_return"] = df["Close"].pct_change()
    for window in [5, 21, 63, 126]:
        df[f"return_{window}d"] = df["Close"].pct_change(window)
    return df

aapl = add_return_features(aapl)
return_cols = ["Date", "Close", "daily_return", "return_5d", "return_21d", "return_63d", "return_126d"]
print(aapl[return_cols].tail(10).round(4))
print(f"\nNull counts:\n{aapl[return_cols].isnull().sum()}")

## 4. Volatility Features

Rolling standard deviation of daily returns, annualised by multiplying
by sqrt(252) — the number of trading days in a year.
High volatility = unpredictable stock = higher risk.

In [ ]:
def add_volatility_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for window in [VOL_SHORT, VOL_LONG]:
        df[f"volatility_{window}d"] = (
            df["daily_return"]
            .rolling(window, min_periods=window)
            .std() * np.sqrt(252)
        )
    return df

aapl = add_volatility_features(aapl)
print(aapl[["Date", "volatility_21d", "volatility_63d"]].tail(10).round(4))

## 5. RSI — Relative Strength Index

RSI measures whether a stock is overbought (RSI > 70) or oversold (RSI < 30).
Uses Wilder smoothing (exponential moving average of gains and losses).
Output is always 0–100.

In [ ]:
def add_rsi(df: pd.DataFrame, period: int = RSI_PERIOD) -> pd.DataFrame:
    df = df.copy()
    delta    = df["Close"].diff()
    gain     = delta.clip(lower=0)
    loss     = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    df["rsi_14"] = 100 - (100 / (1 + rs))
    return df

aapl = add_rsi(aapl)
print(aapl[["Date", "Close", "rsi_14"]].tail(10).round(2))
print(f"\nRSI range: {aapl['rsi_14'].min():.1f} to {aapl['rsi_14'].max():.1f}  (expect 0 to 100)")

## 6. MACD — Moving Average Convergence Divergence

MACD captures trend direction and momentum shifts.
- **MACD line** = fast EMA(12) minus slow EMA(26)
- **Signal line** = EMA(9) of the MACD line
- **Histogram** = MACD minus signal (positive = bullish momentum building)

In [ ]:
def add_macd(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    ema_fast         = df["Close"].ewm(span=MACD_FAST,   adjust=False).mean()
    ema_slow         = df["Close"].ewm(span=MACD_SLOW,   adjust=False).mean()
    df["macd"]       = ema_fast - ema_slow
    df["macd_signal"]= df["macd"].ewm(span=MACD_SIGNAL, adjust=False).mean()
    df["macd_hist"]  = df["macd"] - df["macd_signal"]
    return df

aapl = add_macd(aapl)
print(aapl[["Date", "macd", "macd_signal", "macd_hist"]].tail(10).round(4))

## 7. Price Structure Features

- **BB position** — where price sits within Bollinger Bands (0 = lower, 1 = upper)
- **BB width** — how wide the bands are (proxy for current volatility regime)
- **52-week position** — where price sits within its yearly high/low range
- **Price vs SMA50/200** — how far price has deviated from its moving averages

In [ ]:
def add_price_structure(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Bollinger Bands
    sma_bb   = df["Close"].rolling(BB_WINDOW).mean()
    std_bb   = df["Close"].rolling(BB_WINDOW).std()
    upper_bb = sma_bb + BB_STD * std_bb
    lower_bb = sma_bb - BB_STD * std_bb
    band_width = upper_bb - lower_bb
    df["bb_position"] = (df["Close"] - lower_bb) / band_width.replace(0, np.nan)
    df["bb_width"]    = band_width / sma_bb.replace(0, np.nan)

    # 52-week position
    rolling_high = df["Close"].rolling(252, min_periods=126).max()
    rolling_low  = df["Close"].rolling(252, min_periods=126).min()
    price_range  = (rolling_high - rolling_low).replace(0, np.nan)
    df["week52_position"] = (df["Close"] - rolling_low) / price_range

    # Price vs SMAs
    df["price_vs_sma50"]  = df["Close"] / df["Close"].rolling(50).mean()  - 1
    df["price_vs_sma200"] = df["Close"] / df["Close"].rolling(200).mean() - 1

    return df

aapl = add_price_structure(aapl)
price_cols = ["Date", "bb_position", "bb_width", "week52_position", "price_vs_sma50", "price_vs_sma200"]
print(aapl[price_cols].tail(10).round(4))

## 8. Volume Features

Volume ratio > 2 means today traded at double the normal pace.
Unusual volume often precedes or confirms price moves.

In [ ]:
def add_volume_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    avg_vol = df["Volume"].rolling(VOLUME_WINDOW, min_periods=VOLUME_WINDOW).mean()
    df["volume_ratio"] = df["Volume"] / avg_vol.replace(0, np.nan)
    return df

aapl = add_volume_features(aapl)
print(aapl[["Date", "Volume", "volume_ratio"]].tail(10).round(3))

## 9. Target Variable

The target is what the model predicts:
1. Calculate forward 21-day return for each row (how much the stock gained over the next month)
2. Assign cross-sectional labels per date — which stocks outperformed their peers?
   - **1 = buy** (top third of all stocks that day)
   - **0 = hold** (middle third)
   - **-1 = sell** (bottom third)

> The last 21 rows per ticker will have NaN targets — they have no future to look forward to.
> These are dropped in the cleanup step.

In [ ]:
def add_target(df: pd.DataFrame, forward_days: int = FORWARD_DAYS) -> pd.DataFrame:
    """
    Add forward return column.
    Must be called after all feature calculations.
    Uses future prices — this is intentional for the target only.
    Features never use future data.
    """
    df = df.copy()
    df["forward_return"] = df["Close"].shift(-forward_days) / df["Close"] - 1
    return df


def add_labels(df: pd.DataFrame) -> pd.DataFrame:
    """
    Assign buy/hold/sell labels cross-sectionally (per date across all tickers).
    The model learns relative outperformance, not absolute return.
    """
    df = df.copy()
    df["target"] = (
        df.groupby("Date")["forward_return"]
        .transform(
            lambda x: pd.qcut(
                x.rank(method="first"),
                q=N_QUANTILES,
                labels=[-1, 0, 1]
            ).astype(float)
        )
    )
    return df


aapl = add_target(aapl)
print(aapl[["Date", "Close", "forward_return"]].tail(10).round(4))
print(f"\nNaN forward_return rows (expect ~{FORWARD_DAYS}): {aapl['forward_return'].isna().sum()}")

## 10. Validate AAPL Feature Set

Confirm every feature looks correct before scaling to all 498 tickers.

In [ ]:
print("AAPL feature set shape:", aapl.shape)
print("\nAll columns:", aapl.columns.tolist())

print("\nNull counts per feature:")
nulls = aapl.isnull().sum()
print(nulls[nulls > 0])

check_cols = [
    "daily_return", "return_21d", "volatility_21d",
    "rsi_14", "macd", "bb_position",
    "volume_ratio", "forward_return"
]
print("\nFeature stats:")
print(aapl[check_cols].describe().round(4))

## 11. Scale to All 498 Tickers

Apply all feature functions to every ticker using `groupby.apply()`.
Each ticker is processed independently so rolling windows
never bleed across company boundaries.

In [ ]:
def build_technical_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply all technical feature functions to a single-ticker DataFrame.
    Called inside groupby — each ticker processed independently.
    """
    df = df.sort_values("Date").copy()
    df = add_return_features(df)
    df = add_volatility_features(df)
    df = add_rsi(df)
    df = add_macd(df)
    df = add_price_structure(df)
    df = add_volume_features(df)
    df = add_target(df)
    return df


print(f"Processing {history['Ticker'].nunique()} tickers...")

all_features = (
    history
    .groupby("Ticker", group_keys=False)
    .apply(build_technical_features)
    .reset_index(drop=True)
)

print(f"Done. Shape: {all_features.shape}")
print(f"Tickers   : {all_features['Ticker'].nunique()}")
print("\nTop null rates:")
print(all_features.isnull().mean().sort_values(ascending=False).head(10).round(4))

## 12. Join Fundamentals — Fiscal-Year Aligned

`merge_asof` with `direction="backward"` is the correct way to join annual
financial data to daily price data without data leakage.

For each price row on date D, it finds the most recent fiscal year end
that is less than or equal to D — i.e. the latest report already published.

This is the same logic used in professional quantitative finance pipelines.

In [ ]:
# Select useful financial columns (only those present in the table)
DESIRED_FIN_COLS = [
    "Ticker", "FiscalYearEnd",
    "Total Revenue", "Net Income", "Gross Profit",
    "Operating Income", "EBITDA",
]
available = [c for c in DESIRED_FIN_COLS if c in financials.columns]
fin = financials[available].copy()

# Derive growth and margin features
fin = fin.sort_values(["Ticker", "FiscalYearEnd"])

if "Total Revenue" in fin.columns:
    fin["revenue_growth"]  = fin.groupby("Ticker")["Total Revenue"].pct_change()

if "Net Income" in fin.columns:
    fin["net_income_growth"] = fin.groupby("Ticker")["Net Income"].pct_change()

if "Gross Profit" in fin.columns and "Total Revenue" in fin.columns:
    fin["gross_margin"] = fin["Gross Profit"] / fin["Total Revenue"].replace(0, np.nan)

if "Operating Income" in fin.columns and "Total Revenue" in fin.columns:
    fin["operating_margin"] = fin["Operating Income"] / fin["Total Revenue"].replace(0, np.nan)

# Sort both sides before merge_asof
all_features = all_features.sort_values(["Ticker", "Date"])
fin = fin.sort_values(["Ticker", "FiscalYearEnd"])

# Time-aligned join — no data leakage
all_features = pd.merge_asof(
    all_features,
    fin,
    left_on="Date",
    right_on="FiscalYearEnd",
    by="Ticker",
    direction="backward",
)

print(f"Shape after financials join: {all_features.shape}")
derived = ["revenue_growth", "net_income_growth", "gross_margin", "operating_margin"]
existing = [c for c in derived if c in all_features.columns]
print(f"Derived financial features: {existing}")

## 13. Join Company Info — Categoricals Only

From `all_info` we only use **static categorical** fields — sector and industry.

The numerical ratios in `all_info` (P/E, margins) are point-in-time snapshots
of today's values. Using them for 2020 price rows would be data leakage.
Numerical fundamentals come from `all_financials` instead (correctly time-stamped).

In [ ]:
info_cats = info[["Ticker", "sector", "industry"]].copy()
all_features = all_features.merge(info_cats, on="Ticker", how="left")

print(f"Shape after info join: {all_features.shape}")
print(f"\nSector distribution:")
print(all_features.drop_duplicates("Ticker")["sector"].value_counts())

## 14. Add Cross-Sectional Labels

Labels are assigned per date across all tickers — the model learns
which stocks outperformed their peers on any given day,
not just which stocks had positive absolute returns.

In [ ]:
all_features = add_labels(all_features)

print("Target distribution:")
print(all_features["target"].value_counts().sort_index())
print(f"\nNull targets (last {FORWARD_DAYS} rows per ticker — expected):")
print(all_features["target"].isna().sum())

## 15. Drop Burn-in Rows & Clean Up

Drop rows where core features are NaN due to rolling window startup.
The longest window is 200 days (SMA200), so the first ~200 rows per
ticker will have NaN features. These rows cannot be used for training.

Also drop the last `FORWARD_DAYS` rows per ticker which have NaN targets.

In [ ]:
REQUIRED_COLS = [
    "daily_return", "return_21d", "volatility_21d",
    "rsi_14", "macd", "bb_position",
    "volume_ratio", "forward_return", "target",
]

rows_before = len(all_features)
all_features = all_features.dropna(subset=REQUIRED_COLS)
rows_after  = len(all_features)

pct_dropped = (rows_before - rows_after) / rows_before * 100
print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Dropped     : {rows_before - rows_after:,}  ({pct_dropped:.1f}%)")
print(f"\nFinal shape : {all_features.shape}")
print(f"Tickers     : {all_features['Ticker'].nunique()}")
print(f"Date range  : {all_features['Date'].min().date()} to {all_features['Date'].max().date()}")

## 16. Final Feature Store Validation

In [ ]:
print("=" * 60)
print("FEATURE STORE QUALITY REPORT")
print("=" * 60)

# Null rates
null_rates  = all_features.isnull().mean().sort_values(ascending=False)
high_null   = null_rates[null_rates > 0.05]
print(f"\nColumns with >5% nulls ({len(high_null)}):")
if not high_null.empty:
    print(high_null.to_string())
else:
    print("  None — clean!")

# Target balance
print("\nTarget balance (expect roughly equal thirds):")
print(all_features["target"].value_counts(normalize=True).sort_index().round(3))

# Duplicate check
dupes = all_features.duplicated(["Ticker", "Date"]).sum()
print(f"\nDuplicate Ticker+Date rows: {dupes}  (expect 0)")

# ML feature list
non_feature_cols = {
    "Ticker", "Date", "Open", "High", "Low", "Close", "Volume",
    "forward_return", "FiscalYearEnd", "target",
    "Total Revenue", "Net Income", "Gross Profit", "Operating Income", "EBITDA"
}
feature_cols = [c for c in all_features.columns if c not in non_feature_cols]
print(f"\nML features ({len(feature_cols)}):")
for f in feature_cols:
    print(f"  {f}")

assert dupes == 0,                              "FAIL: duplicate rows found"
assert all_features["target"].isna().sum() == 0, "FAIL: null targets remain"
assert "rsi_14" in all_features.columns,        "FAIL: rsi_14 missing"
assert "sector" in all_features.columns,        "FAIL: sector missing"
print("\n✓ All assertions passed")
print("=" * 60)

## 17. Save Feature Store

In [ ]:
all_features.to_parquet(FEATURES_FILE, index=False, compression="snappy")

ts       = datetime.now().strftime("%Y%m%d_%H%M")
versioned = PROCESSED_DIR / f"features_{ts}.parquet"
all_features.to_parquet(versioned, index=False, compression="snappy")

size_mb = FEATURES_FILE.stat().st_size / 1_048_576
logger.info(f"Feature store saved — {all_features.shape} — {size_mb:.2f} MB")
print(f"✓  features.parquet           {len(all_features):>8,} rows x {len(all_features.columns):>3} cols  ({size_mb:.2f} MB)")
print(f"✓  features_{ts}.parquet  (versioned snapshot)")

## 18. Smoke Test — Reload & Verify

Always reload from disk after saving. Catches silent write failures.

In [ ]:
df_check = pd.read_parquet(FEATURES_FILE)

assert "AAPL"   in df_check["Ticker"].values,   "AAPL missing"
assert "target" in df_check.columns,            "target column missing"
assert "rsi_14" in df_check.columns,            "rsi_14 missing"
assert df_check["target"].isna().sum() == 0,    "null targets on reload"

print(f"Reloaded shape : {df_check.shape}")
print(f"Columns        : {df_check.columns.tolist()}")
print("\n✓ Smoke test passed — feature store reloads cleanly")

## 19. Summary & Next Steps

In [ ]:
non_feature_cols = {
    "Ticker", "Date", "Open", "High", "Low", "Close", "Volume",
    "forward_return", "FiscalYearEnd", "target",
    "Total Revenue", "Net Income", "Gross Profit", "Operating Income", "EBITDA"
}
feature_cols = [c for c in all_features.columns if c not in non_feature_cols]

print("=" * 60)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 60)
print(f"  Rows          : {len(all_features):,}")
print(f"  Columns       : {len(all_features.columns)}")
print(f"  Tickers       : {all_features['Ticker'].nunique()}")
print(f"  ML features   : {len(feature_cols)}")
print(f"  Target        : {FORWARD_DAYS}d forward return bucketed into buy / hold / sell")
print()
print("Output: data/processed/features.parquet")
print()
print("Next notebook : 03_eda.ipynb")
print("  Will analyse: feature distributions, correlations,")
print("  target balance, feature vs target signal strength")
print("=" * 60)